# Notebook 06 — Paired Delta Analysis Across Phases

Computes paired (per-seed) ΔRMSE and ΔR² between every interesting pair of phases.

Why paired and not unpaired: on small datasets like ESOL, the dominant noise source is the scaffold split itself (per-seed variance ~0.13 RMSE in Phase 1). Same seed → same scaffold split, so the variance of (B − A) for the same seed cancels that noise. The result is a tighter, more interpretable Δ than `mean(B) − mean(A)`.

Reads phase summary JSONs produced by the `save_summary_json` pattern (see `src/reporting.py`). Phases 1 and 2 were backfilled post-hoc with the same schema, see commit `P1 backfill` / `P2 backfill`.

**Outputs:**
- Console: comparison table + per-seed detail
- `reports/paired_deltas.md` — drop-in markdown for the README
- `reports/paired_deltas.json` — structured data for downstream automation

When a new phase summary lands (e.g. `phase5_chemprop`), add its name to `PHASES_TO_LOAD` and the relevant pairs to `PAIRS`. The notebook then regenerates all deltas including those involving the new phase.

## 1. Setup

In [ ]:
import os
import sys
import json
from pathlib import Path

# Detect runtime environment
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/qsar-esol-solubility')
else:
    PROJECT_ROOT = Path.cwd().parent

REPORTS_DIR = PROJECT_ROOT / 'reports'

# Make src/ importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.reporting import load_summary_json, paired_delta

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Reports dir : {REPORTS_DIR}")

## 2. Load available phase summaries

Skips phases whose JSON doesn't exist yet (so the notebook still runs partway before Phase 5 is finished).

In [ ]:
PHASES_TO_LOAD = ['phase1', 'phase2', 'phase3', 'phase4']  # add 'phase5' after ChemProp

summaries = {}
for phase in PHASES_TO_LOAD:
    path = REPORTS_DIR / f'{phase}_summary.json'
    if path.exists():
        summaries[phase] = load_summary_json(path)
        agg = summaries[phase]['aggregate']
        n_seeds = len(summaries[phase]['per_seed'])
        print(f"✓ {phase}: {n_seeds} seeds, "
              f"RMSE={agg['RMSE_mean']:.3f}±{agg['RMSE_std']:.3f}, "
              f"R²={agg['R2_mean']:+.3f}±{agg['R2_std']:.3f}")
    else:
        print(f"✗ {phase}: {path} not found — skipping")

print(f"\nLoaded {len(summaries)} summaries: {sorted(summaries.keys())}")

## 3. Compute paired deltas

Sign convention (from `paired_delta` docstring): negative ΔRMSE is improvement, positive ΔR² is improvement. The `wins` counter reflects the per-seed direction (how many of the 5 seeds Phase B beats Phase A in the appropriate direction).

Pair design:
- **P1 → P2**: featurization effect, same model (untuned RF). Isolates the value of RDKit descriptors.
- **P2 → P3**: model effect, same features (Morgan + desc). Isolates the value of XGBoost + Optuna over untuned RF.
- **P1 → P3**: cumulative (features + model + tuning). The full Phase 1 → 3 trajectory.
- **P1 → P4**: model class effect on a fixed featurization (Morgan only). Isolates expressivity gain from MLP over RF without the feature confound. This replaces the unpaired Δ currently in the README.
- **P2 → P4** and **P3 → P4**: how much MLP Morgan-only sacrifices vs phases that get the global descriptors. Expected to be positive (MLP loses) — this measures the cost of the feature ablation.

Skip pairs where either phase isn't loaded (so the notebook still produces partial output).

In [ ]:
PAIRS = [
    ('phase1', 'phase2', 'P1 → P2', 'add RDKit descriptors (same RF)'),
    ('phase2', 'phase3', 'P2 → P3', 'tune XGBoost (same features)'),
    ('phase1', 'phase3', 'P1 → P3', 'tuned XGB + desc vs untuned RF + Morgan'),
    ('phase1', 'phase4', 'P1 → P4', 'tuned MLP vs untuned RF (Morgan-only both)'),
    ('phase2', 'phase4', 'P2 → P4', 'MLP Morgan-only vs RF Morgan+desc'),
    ('phase3', 'phase4', 'P3 → P4', 'MLP Morgan-only vs XGB Morgan+desc'),
    # When P5 lands, add e.g.:
    # ('phase4', 'phase5', 'P4 → P5', 'ChemProp D-MPNN vs MLP (Morgan-only baseline)'),
    # ('phase3', 'phase5', 'P3 → P5', 'ChemProp vs XGBoost (best feature-engineering baseline)'),
]

deltas = {}
for a, b, label, note in PAIRS:
    if a not in summaries or b not in summaries:
        print(f"⏭  {label}: missing {a} or {b}")
        continue
    d_rmse = paired_delta(summaries[a]['per_seed'], summaries[b]['per_seed'], 'RMSE')
    d_r2   = paired_delta(summaries[a]['per_seed'], summaries[b]['per_seed'], 'R2')
    deltas[label] = {
        'rmse': d_rmse, 'r2': d_r2,
        'note': note,
        'phase_a': a, 'phase_b': b,
    }
    print(f"\n{label}  ({note})")
    print(f"  ΔRMSE = {d_rmse['mean_delta']:+.3f} ± {d_rmse['std_delta']:.3f}  "
          f"({d_rmse['wins']}/{d_rmse['n_seeds']} seeds improved)")
    print(f"  ΔR²   = {d_r2['mean_delta']:+.3f} ± {d_r2['std_delta']:.3f}  "
          f"({d_r2['wins']}/{d_r2['n_seeds']} seeds improved)")

## 4. Summary table

Compact comparison view. Negative ΔRMSE = Phase B wins; positive ΔR² = Phase B wins.

In [ ]:
print("=" * 78)
print(f"{'Comparison':<12} {'ΔRMSE':>18} {'ΔR²':>18} {'RMSE wins':>11} {'R² wins':>10}")
print("=" * 78)
for label, d in deltas.items():
    r = d['rmse']; r2 = d['r2']
    print(f"{label:<12} "
          f"{r['mean_delta']:+.3f} ± {r['std_delta']:.3f}    "
          f"{r2['mean_delta']:+.3f} ± {r2['std_delta']:.3f}    "
          f"{r['wins']}/{r['n_seeds']:>5}  {r2['wins']}/{r2['n_seeds']:>5}")
print("=" * 78)

## 5. Per-seed detail (sanity check)

If a pair shows a counterintuitive result (e.g. 1/5 wins despite a favorable mean delta), the per-seed table tells you which seed is the outlier. Often it's the same seed that had a degenerate test set in some other phase.

In [ ]:
SEEDS = [42, 0, 1, 2, 3]
print("Per-seed RMSE deltas (negative = Phase B improvement)")
print("-" * 60)
print(f"{'Pair':<12} " + " ".join(f"{s:>9}" for s in SEEDS))
for label, d in deltas.items():
    per_seed = {x['seed']: x['delta'] for x in d['rmse']['per_seed_delta']}
    row = " ".join(f"{per_seed.get(s, float('nan')):>+9.3f}" for s in SEEDS)
    print(f"{label:<12} {row}")

## 6. Generate markdown block for README

Writes a copy-paste-ready table to `reports/paired_deltas.md`, and also dumps the full structured data to `reports/paired_deltas.json` for downstream automation (e.g. blog post auto-gen, or a future CI check that flags regressions).

In [ ]:
md_lines = []
md_lines.append("**Paired delta summary (5-seed scaffold, same split per seed):**\n")
md_lines.append("| Comparison | ΔRMSE | ΔR² | RMSE wins | R² wins | Note |")
md_lines.append("|---|---|---|---|---|---|")
for label, d in deltas.items():
    r = d['rmse']; r2 = d['r2']
    md_lines.append(
        f"| {label} | "
        f"{r['mean_delta']:+.3f} ± {r['std_delta']:.3f} | "
        f"{r2['mean_delta']:+.3f} ± {r2['std_delta']:.3f} | "
        f"{r['wins']}/{r['n_seeds']} | "
        f"{r2['wins']}/{r2['n_seeds']} | "
        f"{d['note']} |"
    )
md_block = "\n".join(md_lines)
print(md_block)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
(REPORTS_DIR / 'paired_deltas.md').write_text(md_block + "\n")
print(f"\n→ saved: {REPORTS_DIR / 'paired_deltas.md'}")

deltas_json_path = REPORTS_DIR / 'paired_deltas.json'
deltas_json_path.write_text(json.dumps(
    {label: {
        'phase_a': d['phase_a'],
        'phase_b': d['phase_b'],
        'note': d['note'],
        'rmse': d['rmse'],
        'r2': d['r2'],
    } for label, d in deltas.items()},
    indent=2,
))
print(f"→ saved: {deltas_json_path}")